# Acoustic Interferometry Data Analysis

Example Colab notebook for analysis of remote acoustic interferometry data collected from [acoustic2.pchem.dev](https://acoustic2.pchem.dev/)

*Prof. Jeffery L. Yarger* (jyarger@proton.me)

March 30, 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy import stats
from scipy.stats import t
import ipywidgets as widgets
from IPython.display import display

# Global variables to store data, url, and valid peaks
default_url = 'https://raw.githubusercontent.com/CHM343/Data/refs/heads/main/Argon_Gas_acoustic2_pchem_dev_WN_22kHz_100vol_Spectrum.csv'
current_url = default_url
current_valid_peaks = []

# Initialize with default data
try:
    spectrum_data = pd.read_csv(current_url, header=0)
    x = spectrum_data['Frequency (Hz)']
    y = spectrum_data['FFT Magnitude (dB)']
except:
    x = pd.Series(dtype=float)
    y = pd.Series(dtype=float)

# Function to plot with dynamic parameters
def plot_peaks(data_url, min_height, width, distance, prominence, min_freq, max_freq):
    global current_valid_peaks, current_url, x, y

    # Load new data only if URL changes
    if data_url != current_url:
        try:
            new_data = pd.read_csv(data_url, header=0)
            x = new_data['Frequency (Hz)']
            y = new_data['FFT Magnitude (dB)']
            current_url = data_url
        except Exception as e:
            print(f"Error loading data from URL: {e}")
            return

    if len(x) == 0 or len(y) == 0:
        print("No valid data to plot.")
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(x, y, linewidth=0.1, alpha=0.5, color='black', label='Signal')

    # Find peaks based on the slider values
    peaks, _ = find_peaks(y, rel_height=min_height, width=width, distance=distance, prominence=prominence)

    # Filter the peaks based on the desired frequency range
    current_valid_peaks = [p for p in peaks if min_freq <= x[p] <= max_freq]

    # Plot peaks
    ax.plot(x[current_valid_peaks], y[current_valid_peaks], "x", color='red', label='Peaks', markersize=8)

    # Draw dotted vertical lines for Min and Max Frequency
    ax.axvline(x=min_freq, color='green', linestyle=':', label='Min Freq', linewidth=2)
    ax.axvline(x=max_freq, color='blue', linestyle=':', label='Max Freq', linewidth=2)

    # Add the Node # integer value above the red x symbols
    for i, p in enumerate(current_valid_peaks):
        # Use annotate to place the text slightly above the point
        ax.annotate(str(i + 1), (x[p], y[p]), textcoords="offset points", xytext=(0, 8), ha='center', color='red', fontweight='bold', fontsize=12)

    ax.set_xlim(0, 10000)
    ax.set_ylim(-100, 0)

    # Make font size for the axis labels and numbers significantly larger
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.xlabel("Frequency (Hz)", fontsize=18)
    plt.ylabel("Magnitude (dB)", fontsize=18)

    plt.show()

# Define the widgets explicitly
url_w = widgets.Text(value=default_url, description='Data URL:', layout=widgets.Layout(width='95%'))
min_height_w = widgets.FloatSlider(value=5, min=-5, max=10, step=1.0, description='Height:')
width_w = widgets.FloatSlider(value=300.0, min=1.0, max=500.0, step=10.0, description='Width:')
distance_w = widgets.IntSlider(value=300, min=1, max=500, step=10, description='Distance:')
prominence_w = widgets.FloatSlider(value=5.0, min=0.0, max=20.0, step=0.5, description='Prom: ')
min_freq_w = widgets.FloatSlider(value=200.0, min=0.0, max=1000.0, step=10.0, description='Min Freq:')
max_freq_w = widgets.FloatSlider(value=9000.0, min=500.0, max=22000.0, step=50.0, description='Max Freq:')

# Group the widgets vertically and add right padding to space out the plot
ui = widgets.VBox([min_height_w, width_w, distance_w, prominence_w, min_freq_w, max_freq_w], layout=widgets.Layout(margin='0px 40px 0px 0px'))

# Create the interactive output mapped to the widgets
out = widgets.interactive_output(plot_peaks, {
    'data_url': url_w,
    'min_height': min_height_w,
    'width': width_w,
    'distance': distance_w,
    'prominence': prominence_w,
    'min_freq': min_freq_w,
    'max_freq': max_freq_w
})

# Display the widgets: URL on top, then sliders (left) and the output (right) side-by-side
display(widgets.VBox([url_w, widgets.HBox([ui, out])]))
print("Note: Run the next cell if you want to display the table of peaks and the Node # vs Frequency plot.")

Note: Run the next cell if you want to display the table of peaks and the Node # vs Frequency plot.


In [ ]:
# Run this cell to optionally display the interactive peak table and Node # vs Frequency plot

if 'current_valid_peaks' in globals() and len(current_valid_peaks) > 0:
    # Create a header for our interactive table
    header = widgets.HBox([
        widgets.Label('Use?', layout=widgets.Layout(width='50px', font_weight='bold')),
        widgets.Label('Node #', layout=widgets.Layout(width='60px', font_weight='bold')),
        widgets.Label('Freq (Hz)', layout=widgets.Layout(width='100px', font_weight='bold')),
        widgets.Label('Mag (dB)', layout=widgets.Layout(width='100px', font_weight='bold'))
    ])

    rows = [header]
    checkboxes = {}

    # Create a row with a checkbox for each peak
    for i, p in enumerate(current_valid_peaks):
        cb = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width='50px'))
        checkboxes[f'cb_{i}'] = cb
        row = widgets.HBox([
            cb,
            widgets.Label(str(i + 1), layout=widgets.Layout(width='60px')),
            widgets.Label(f"{x[p]:.2f}", layout=widgets.Layout(width='100px')),
            widgets.Label(f"{y[p]:.2f}", layout=widgets.Layout(width='100px'))
        ])
        rows.append(row)

    # Combine rows into a VBox, add a scrollbar if it gets too long
    table_ui = widgets.VBox(rows, layout=widgets.Layout(margin='0px 40px 0px 0px', max_height='400px', overflow_y='auto'))

    # Create new widgets for Gas and Speed of Sound
    gas_w = widgets.Text(value='Argon', description='Gas:')
    speed_w = widgets.FloatText(value=320.0, step=0.1, description='Speed (m/s):')
    calc_ui = widgets.HBox([gas_w, speed_w])

    # Function to update the plot based on checked boxes
    def update_fit_plot(gas_name, speed_of_sound, **kwargs):
        selected_nodes = []
        selected_freqs = []

        # Check which checkboxes are True
        for i, p in enumerate(current_valid_peaks):
            if kwargs.get(f'cb_{i}', False):
                selected_nodes.append(i + 1)
                selected_freqs.append(x[p])

        fig2, ax2 = plt.subplots(figsize=(6, 4))

        stats_text = ""
        if len(selected_nodes) > 2:
            nodes = np.array(selected_nodes)
            freqs = np.array(selected_freqs)

            # Linear regression
            res = stats.linregress(nodes, freqs)
            slope = res.slope
            intercept = res.intercept
            slope_stderr = res.stderr
            intercept_stderr = res.intercept_stderr

            # Calculate 95% Confidence Intervals
            dof = len(nodes) - 2
            t_val = t.ppf(0.975, dof)
            ci_slope = t_val * slope_stderr
            ci_intercept = t_val * intercept_stderr

            # Scatter Plot with updated styling
            ax2.scatter(nodes, freqs, facecolors='none', edgecolors='red', alpha=0.75, label='Peak Data', s=50)
            ax2.plot(nodes, intercept + slope * nodes, color='black', linestyle='-', alpha=0.95, label='Linear Fit')

            # Update labels and tick sizes
            ax2.set_xlabel('Node #', fontsize=14)
            ax2.set_ylabel('Frequency (Hz)', fontsize=14)
            ax2.tick_params(axis='both', which='major', labelsize=12)

            # Tube length calculation
            if slope > 0:
                L = speed_of_sound / (2 * slope)
                # Fractional error in Length equals fractional error in slope
                ci_L = L * (ci_slope / slope)

                stats_text = (f"Slope: {slope:.2f} ± {ci_slope:.2f} Hz/Node (95% CI)\n"
                              f"Intercept: {intercept:.2f} ± {ci_intercept:.2f} Hz (95% CI)\n"
                              f"\n--- Tube Length Calculation ---\n"
                              f"Gas: {gas_name}, Speed of Sound: {speed_of_sound} m/s\n"
                              f"Calculated Length (L): {L:.4f} ± {ci_L:.4f} m (95% CI)")
            else:
                stats_text = (f"Slope: {slope:.2f} ± {ci_slope:.2f} Hz/Node (95% CI)\n"
                              f"Intercept: {intercept:.2f} ± {ci_intercept:.2f} Hz (95% CI)\n"
                              f"Slope must be positive to calculate length.")
            ax2.legend()
        else:
            ax2.text(0.5, 0.5, 'Need at least 3 points\nfor a valid linear fit.',
                     ha='center', va='center', fontsize=12, color='red')
            ax2.set_axis_off()

        plt.tight_layout()
        plt.show()

        # Print the stats below the plot
        if stats_text:
            print(stats_text)

    # Map checkboxes and calc inputs to the plotting function
    controls = {'gas_name': gas_w, 'speed_of_sound': speed_w}
    controls.update(checkboxes)
    out = widgets.interactive_output(update_fit_plot, controls)

    # Display table and plot side-by-side (with inputs above the plot)
    right_panel = widgets.VBox([calc_ui, out])
    display(widgets.HBox([table_ui, right_panel]))

else:
    print("No peaks found. Adjust the sliders in the previous cell to find peaks first.")